Data sources:

- https://www.nar.realtor/research-and-statistics/housing-statistics/county-median-home-prices-and-monthly-mortgage-payment
- https://taxfoundation.org/data/all/state/property-taxes-by-state-county-2023/
- https://www.sycamoreequity.com/housing-affordability-county-table
- https://gis.cancer.gov/tools/urban-sprawl/
- https://www.cdc.gov/nchs/data-analysis-tools/urban-rural.html
- 

In [62]:
import pandas as pd
import zipfile

In [20]:
df=pd.read_excel('natamenity.xlsx')
df['STCOFIPS']=df['FIPS'].astype(str).str.zfill(5)

df_combined=df[['STCOFIPS', 'nat_amenity_score', 'mean_temp_jan', 'mean_sun_jan', 'mean_temp_jul', 'mean_hum_jul', 'land_surface',
       'water_perc']]

In [29]:
df=pd.read_csv('urbanism_cdc.csv')
df['STCOFIPS']=df['FIPS code'].astype(str).str.zfill(5)
df=df.rename(columns={'2023 Code':'urbanism'})

In [31]:
df_combined=df_combined.merge(df[['STCOFIPS', 'FullGeoName', 'County_name', 'urbanism']], how='outer')

In [33]:
df=pd.read_excel('Median_property_taxes_by_county_2023_Tax_Foundation.xlsx')

df['STCOFIPS']=df['geo_id'].astype(str).str[-5:]

In [42]:
df_combined=df_combined.merge(df[['STCOFIPS', 'real_estate_taxes_med']], how='outer')

In [54]:
df=pd.read_csv('housing_afford.csv')

df['STCOFIPS']=df['FIPS'].astype(str).str.zfill(5)

df=df.rename(columns={'2017':'med_home_price2017',
                      '2023 Ratio':'income_price_ratio2023'})

df_combined=df_combined.merge(df[['STCOFIPS', 'med_home_price2017', 'income_price_ratio2023']], how='outer')

In [59]:
df_combined.to_parquet('../urban_afford.parquet')

In [78]:
zip_path="../ACSDT5Y2016.B19080_2025-12-05T143905.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    first_file = z.namelist()[0]
    print("Reading:", first_file)

    with z.open(first_file) as f:
        df = pd.read_csv(f, encoding="latin1", skiprows=1)

Reading: ACSDT5Y2016.B19080-Data.csv


In [79]:
df['q20'] = pd.to_numeric(
    df['Estimate!!Quintile Upper Limits!!Lowest Quintile'],
    errors='coerce'
)

df['q80'] = pd.to_numeric(
    df['Estimate!!Quintile Upper Limits!!Fourth Quintile'],
    errors='coerce'
)

df['inequality_ratio']=df['q80']/df['q20']

df['STCOFIPS']=df['Geography'].astype(str).str[-5:]

In [80]:
df

,Geography,Geographic Area Name,Estimate!!Quintile Upper Limits!!Lowest Quintile,Margin of Error!!Quintile Upper Limits!!Lowest Quintile,Estimate!!Quintile Upper Limits!!Second Quintile,Margin of Error!!Quintile Upper Limits!!Second Quintile,Estimate!!Quintile Upper Limits!!Third Quintile,Margin of Error!!Quintile Upper Limits!!Third Quintile,Estimate!!Quintile Upper Limits!!Fourth Quintile,Margin of Error!!Quintile Upper Limits!!Fourth Quintile,Estimate!!Lower Limit of Top 5 Percent,Margin of Error!!Lower Limit of Top 5 Percent,Unnamed: 12,q20,q80,inequality_ratio,STCOFIPS
0,0500000US01001,"Autauga County, Alabama",23134,2101,41803,2534,66372,3489,101520,4465,161080,13607,NaN,23134.0,101520,4.388346,01001
1,0500000US01003,"Baldwin County, Alabama",22306,923,40762,879,63200,1922,102624,2929,184785,12542,NaN,22306.0,102624,4.600735,01003
2,0500000US01005,"Barbour County, Alabama",12650,1034,25159,1970,42331,2588,74150,3702,120688,8568,NaN,12650.0,74150,5.861660,01005
3,0500000US01007,"Bibb County, Alabama",18893,2456,32712,2494,49645,4157,79882,8528,132789,14935,NaN,18893.0,79882,4.228127,01007
4,0500000US01009,"Blount County, Alabama",20634,1398,36394,1771,56421,3420,83944,4175,136941,11600,NaN,20634.0,83944,4.068237,01009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3215,0500000US72145,"Vega Baja Municipio, Puerto Rico",4930,976,12845,1136,23534,1975,44683,3971,76190,5394,NaN,4930.0,44683,9.063489,72145
3216,0500000US72147,"Vieques Municipio, Puerto Rico",9176,1508,15423,1545,24140,4878,38373,6758,56998,13136,NaN,9176.0,38373,4.181888,72147
3217,0500000US72149,"Villalba Municipio, Puerto Rico",7627,842,15155,1189,25084,1444,37209,1738,70156,7595,NaN,7627.0,37209,4.878589,72149
3218,0500000US72151,"Yabucoa Municipio, Puerto Rico",5462,1131,12737,1109,22088,1798,35451,2961,59245,3179,NaN,5462.0,35451,6.490480,72151


In [75]:
df_combined=df_combined.merge(df[['STCOFIPS', 'inequality_ratio']], how='outer')

In [76]:
df_combined.to_parquet('../urban_afford.parquet')